# Tim tham so - Uncertainty-Aware UCB-VAE

Tuning hai giai doan VAE -> UCB, chia bon batch Kaggle, giu reward/cost network/frozen standardization tu Battle of 3 Variants.


## Cách chạy trên bốn tài khoản Kaggle

1. Upload cùng notebook và repository lên cả bốn tài khoản.
2. Giữ `TUNING_STAGE="VAE"`, đặt `RUN_TUNING=True`; mỗi tài khoản đặt `BATCH_IDX` lần lượt `0`, `1`, `2`, `3`.
3. Mỗi tài khoản tự lưu CSV, JSONL, leaderboard và biểu đồ của batch đang chạy. Notebook không tự tìm hay gộp output từ máy khác.
4. Sau khi bạn xác định VAE tốt nhất từ kết quả bốn máy, chép tham số đó vào `LOCKED_VAE_CONFIG`.
5. Đổi `TUNING_STAGE="UCB"` và lặp lại bốn batch để tìm β-UCB.

Test chính thức 2022–2023 không được truyền vào hàm chấm điểm trial. Sau khi khóa VAE và UCB, retrain trên toàn bộ Train 2013–2021 rồi đánh giá 20 seed trên Test 2022–2023 trong notebook Battle.

In [ ]:
!git clone https://github.com/kohi-vip/SARSA_FinancialRL.git

In [ ]:
!pip install numpy pandas matplotlib tqdm torch TA-Lib optuna

In [ ]:
# BLOCK 1 — Cấu hình tìm siêu tham số Uncertainty-Aware UCB-VAE trên 4 tài khoản Kaggle
from __future__ import annotations

import gc
import json
import math
import random
import time
import traceback
from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

# Mỗi tài khoản dùng cùng notebook và chỉ đổi BATCH_IDX thành 0, 1, 2 hoặc 3.
BATCH_IDX = 0
NUM_BATCHES = 4

# Chạy VAE trước. Sau khi gộp đủ 4 batch và chọn VAE, đổi thành "UCB".
TUNING_STAGE = "VAE"  # "VAE" hoặc "UCB"
RUN_TUNING = False
RESUME = True
SCREENING_SEEDS = (42, 43, 44)

# Khi TUNING_STAGE="UCB", chép top VAE từ file best_vae_config.json vào đây.
LOCKED_VAE_CONFIG = {
    "vae_latent_dim": 16,
    "vae_lr": 1e-3,
    "vae_beta_kl": 0.01,
    "vae_batch_size": 256,
    "bootstrap_trajectories": 5,
    "bootstrap_updates": 100,
    "online_aux_updates": 1,
    "vae_replay_capacity": 50_000,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA = 0.95
NN_LR = 5e-5
COST_LR = 1e-3
EPISODES = 45
SARSA_ALPHA = 0.60
Q_WEIGHT_DECAY = 1e-4
BATCH_SIZE = 128
LATENT_DIM = 16
BALANCE_INIT = 1_000.0
TRANSACTION_FEE = 0.001
W_RISK = 0.15
W_STABILITY = 0.05
ZETA = 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT = 2.0
SCALER_SEED = 43
CURRENT_RUN_SEED = SCREENING_SEEDS[0]

# VAE: 3 × 4 × 4 = 48 cấu hình → 12 cấu hình/tài khoản.
VAE_GRID = []
for latent_dim, vae_lr, beta_kl in product(
    (8, 16, 32),
    (1e-4, 3e-4, 1e-3, 3e-3),
    (1e-4, 1e-3, 1e-2, 5e-2),
):
    VAE_GRID.append({
        "vae_latent_dim": latent_dim,
        "vae_lr": vae_lr,
        "vae_beta_kl": beta_kl,
        "vae_batch_size": 256,
        "bootstrap_trajectories": 5,
        "bootstrap_updates": 100,
        "online_aux_updates": 1,
        "vae_replay_capacity": 50_000,
    })

# UCB: 7 × 4 × 2 = 56 cấu hình → 14 cấu hình/tài khoản.
UCB_GRID = []
for beta_0, beta_decay, beta_min in product(
    (0.01, 0.03, 0.05, 0.10, 0.15, 0.30, 0.50),
    (0.90, 0.93, 0.96, 0.99),
    (0.005, 0.01),
):
    UCB_GRID.append({"beta_0": beta_0, "beta_decay": beta_decay, "beta_min": beta_min})

BASE_CONFIG = {
    "label": "Uncertainty-Aware UCB-VAE",
    "use_cost": True,
    "robust_loss": True,
    "weight_decay": Q_WEIGHT_DECAY,
    "reward_shaping": True,
    "kl_reduction": "sum",
    "beta_0": 0.03,
    "beta_decay": 0.91,
    "beta_min": 0.01,
    **LOCKED_VAE_CONFIG,
}

if TUNING_STAGE not in {"VAE", "UCB"}:
    raise ValueError("TUNING_STAGE phải là 'VAE' hoặc 'UCB'.")
if not 0 <= BATCH_IDX < NUM_BATCHES:
    raise ValueError(f"BATCH_IDX phải nằm trong [0, {NUM_BATCHES - 1}].")

FULL_GRID = VAE_GRID if TUNING_STAGE == "VAE" else UCB_GRID
for index, candidate in enumerate(FULL_GRID):
    candidate["config_id"] = f"{TUNING_STAGE.lower()}_{index:03d}"
BATCH_CONFIGS = [candidate for index, candidate in enumerate(FULL_GRID) if index % NUM_BATCHES == BATCH_IDX]

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print({
    "device": str(DEVICE), "stage": TUNING_STAGE, "batch_idx": BATCH_IDX,
    "configs_this_account": len(BATCH_CONFIGS), "runs_this_account": len(BATCH_CONFIGS) * len(SCREENING_SEEDS),
    "full_grid": len(FULL_GRID), "seeds": SCREENING_SEEDS,
})

In [ ]:
# BLOCK 2 — HPG, reward cải tiến và Validation chỉ nằm trong Train chính thức
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
OUTPUT_DIR = OUTPUT_ROOT / "tim_tham_so" / TUNING_STAGE.lower()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]
STATE_NAMES = ["Price", "Balance", "Position", "MACD", "RSI", "CCI", "ADX"]

def load_hpg_bad() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing: raise ValueError(f"HPG BAD {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        frame[REQUIRED_COLUMNS[1:]] = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if frame[REQUIRED_COLUMNS[1:]].isna().any().any():
            raise ValueError(f"HPG BAD {label} chứa NaN/giá trị không hợp lệ.")
    if train["time"].max() >= test["time"].min():
        raise ValueError("Train/Test chồng lấn thời gian.")
    return train.reset_index(drop=True), test.reset_index(drop=True)

def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray([row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]], dtype=np.float32)

class TradingEnv:
    """Giữ nguyên môi trường và reward cải tiến của Battle of 3 Variants."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2: raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True); self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash); self.reset()

    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash; self.portfolio_history = [self.initial_cash]; self.var_targets = []
        return self._state()

    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)

    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            affordable = int(self.cash // (price * (1.0 + TRANSACTION_FEE)))
            executed = min(int(requested), affordable)
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed

    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping:
            reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value)); self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {"raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return), "var_target": float(var_target),
                "drawdown": float(drawdown), "executed_action": int(executed), "portfolio_value": float(portfolio_value)}
        return self._state(), float(reward), done, info

@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, states: np.ndarray) -> np.ndarray:
        return (np.asarray(states, dtype=np.float32) - self.mean) / self.std

def calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    rng = np.random.default_rng(SCALER_SEED); states = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False); state, done = env.reset(), False
        while not done:
            states.append(state.copy()); state, _, done, _ = env.step(int(rng.choice(ACTION_VALUES)))
    return np.asarray(states, dtype=np.float32)

official_train_hpg, official_test_hpg = load_hpg_bad()
# Validation theo thời gian chỉ phục vụ tuning; Test 2022–2023 tuyệt đối không dùng để xếp hạng trial.
validation_start = pd.Timestamp("2019-01-01")
tune_train_hpg = official_train_hpg[official_train_hpg["time"] < validation_start].reset_index(drop=True)
valid_hpg = official_train_hpg[official_train_hpg["time"] >= validation_start].reset_index(drop=True)
if len(tune_train_hpg) < 2 or len(valid_hpg) < 2:
    split = int(len(official_train_hpg) * 0.67)
    tune_train_hpg = official_train_hpg.iloc[:split].reset_index(drop=True)
    valid_hpg = official_train_hpg.iloc[split:].reset_index(drop=True)
train_hpg = tune_train_hpg
calibration = calibration_states(tune_train_hpg)
frozen_mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenScaler(frozen_mean.copy(), frozen_std.copy())
FROZEN_SCALER.mean.setflags(write=False); FROZEN_SCALER.std.setflags(write=False)
print({
    "tuning_train": (str(tune_train_hpg.time.min().date()), str(tune_train_hpg.time.max().date()), len(tune_train_hpg)),
    "validation": (str(valid_hpg.time.min().date()), str(valid_hpg.time.max().date()), len(valid_hpg)),
    "untouched_test": (str(official_test_hpg.time.min().date()), str(official_test_hpg.time.max().date()), len(official_test_hpg)),
})

In [ ]:
# Ô CODE 3 — Q-Network, EpistemicVAE và Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class EpistemicVAE(nn.Module):
    """Joint VAE: 7 state + 11 action one-hot → latent 16 → reconstruction 18."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        reconstructed = self.decoder(mu + torch.randn_like(std) * std)
        return reconstructed, mu, logvar

    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        """Novelty với KLD sum/mean tùy ablation và probe z⁺=μ+1.96σ."""
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl_terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum":
            kl = torch.sum(kl_terms, dim=-1)
        elif kl_reduction == "mean":
            kl = torch.mean(kl_terms, dim=-1)
        else:
            raise ValueError("kl_reduction phải là 'sum' hoặc 'mean'.")
        std = torch.exp(0.5 * logvar)
        reconstructed_95 = self.decoder(mu + 1.96 * std)
        reconstruction_error = torch.norm(x - reconstructed_95, p=2, dim=-1)
        return kl + reconstruction_error


class CostNetwork(nn.Module):
    """Ước lượng VaR không âm từ concat(s_scaled, action_onehot) ∈ R^18."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)


def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE(), "\n", CostNetwork())


In [ ]:
# BLOCK 4 — Loss, replay và Confidence-Weighted Risk Penalty (tham số hóa cho grid)
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)

def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl, recon, kl

def cost_loss(predicted_var, var_target):
    return F.huber_loss(predicted_var, var_target)

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = int(capacity); self.states = []; self.actions = []; self.var_targets = []
    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32)); self.actions.append(int(action)); self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]; del self.actions[:overflow]; del self.var_targets[:overflow]
    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0: raise RuntimeError("Không thể sample replay buffer rỗng.")
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return (np.asarray([self.states[i] for i in indices]), np.asarray([self.actions[i] for i in indices]),
                np.asarray([self.var_targets[i] for i in indices], dtype=np.float32))
    def __len__(self): return len(self.states)

def action_scores(q_network, vae, cost_network, state, beta, config, collect_details=False):
    scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1); action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval(); cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        predicted_var = cost_network(state_batch, action_batch)
        confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
        penalty = torch.where(predicted_var < ZETA, torch.zeros_like(predicted_var), confidence * predicted_var)
        scores = q_values - penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None if not collect_details else {
        "novelty": novelty_raw.detach().cpu().numpy(), "predicted_var": predicted_var.detach().cpu().numpy(),
        "penalty": penalty.detach().cpu().numpy(),
    }
    return int(ACTION_VALUES[action_index]), action_index, details

def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config):
    raw_states, action_indices, var_targets = replay.sample(config["vae_batch_size"])
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE); actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions); target = torch.cat([states, actions], dim=-1)
    loss_vae, recon, kl = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True); loss_vae.backward(); torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0); vae_optimizer.step()
    targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
    predicted = cost_network(states, actions); loss_c = cost_loss(predicted, targets)
    cost_optimizer.zero_grad(set_to_none=True); loss_c.backward(); torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0); cost_optimizer.step()
    return float(loss_vae.detach().cpu()), float(recon.detach().cpu()), float(kl.detach().cpu()), float(loss_c.detach().cpu())

def random_bootstrap(replay: ReplayBuffer, config):
    rng = np.random.default_rng(CURRENT_RUN_SEED)
    for _ in range(int(config["bootstrap_trajectories"])):
        env = TradingEnv(tune_train_hpg, reward_shaping=False); state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11)); states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index])); var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)

def collect_episode(env, q_network, vae, cost_network, beta, config):
    state, done = env.reset(), False; states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy()); rewards.append(reward)
        actions.append(action_index); dones.append(done); var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def q_update(q_network, optimizer, trajectory, robust: bool):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE); an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE); losses = []; q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return losses

In [ ]:
# BLOCK 5 — Chạy một config/seed chỉ trên Tuning-Train và Validation
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)

def evaluate_policy(q_network, vae, cost_network, beta, config, data, previous_row=None):
    env = TradingEnv(evaluation_frame(data, previous_row), reward_shaping=config["reward_shaping"])
    state, done = env.reset(), False; novelty_values = []
    while not done:
        action, _, details = action_scores(q_network, vae, cost_network, state, beta, config, collect_details=True)
        novelty_values.extend(details["novelty"].tolist()); state, _, done, _ = env.step(action)
    return np.asarray(env.portfolio_history, dtype=np.float64), np.asarray(env.var_targets), np.asarray(novelty_values)

def period_metrics(portfolio: np.ndarray, dates: Sequence[pd.Timestamp], var_targets=None) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0]); roi = float(profit / max(abs(portfolio[0]), 1e-8) * 100.0)
    dates = pd.Series(dates).reset_index(drop=True); days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = float(portfolio[-1] / max(portfolio[0], 1e-8)); arr = float((ratio ** (365.25 / days) - 1.0) * 100.0) if ratio > 0 else -100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else float((annual_return - RISK_FREE_RATE_PERCENT) / volatility)
    peaks = np.maximum.accumulate(portfolio); mdd = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    violations = int(np.sum(np.asarray(var_targets) > ZETA)) if var_targets is not None else 0
    return {"profit": profit, "roi": roi, "arr": arr, "sharpe": sharpe, "max_drawdown": mdd, "violations": violations}

def candidate_config(candidate: Mapping[str, Any]) -> Dict[str, Any]:
    config = dict(BASE_CONFIG); config.update({k: v for k, v in candidate.items() if k != "config_id"})
    return config

def run_config_seed(candidate: Mapping[str, Any], seed: int, progress_bar=None) -> Dict[str, Any]:
    global CURRENT_RUN_SEED
    CURRENT_RUN_SEED = int(seed); set_seed(seed); config = candidate_config(candidate)
    q_network = QNetwork().to(DEVICE); vae = EpistemicVAE(latent_dim=int(config["vae_latent_dim"])).to(DEVICE); cost_network = CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=float(config["vae_lr"])); cost_optimizer = torch.optim.Adam(cost_network.parameters(), lr=COST_LR)
    replay = ReplayBuffer(config["vae_replay_capacity"]); q_losses = []; vae_losses = []; recon_losses = []; kl_losses = []; cost_losses = []
    random_bootstrap(replay, config)
    for _ in range(int(config["bootstrap_updates"])):
        lv, lr, lk, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
        vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)
    for episode in range(EPISODES):
        beta = max(float(config["beta_min"]), float(config["beta_0"]) * float(config["beta_decay"]) ** episode)
        trajectory = collect_episode(TradingEnv(tune_train_hpg, reward_shaping=True), q_network, vae, cost_network, beta, config)
        replay.add(trajectory[0], trajectory[3], trajectory[6]); q_losses.extend(q_update(q_network, q_optimizer, trajectory, True))
        for _ in range(int(config["online_aux_updates"])):
            lv, lr, lk, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
            vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)
        if progress_bar is not None and ((episode + 1) % 5 == 0 or episode + 1 == EPISODES):
            progress_bar.set_postfix(config=candidate["config_id"], seed=seed, episode=f"{episode + 1}/{EPISODES}")
    train_portfolio, train_var, train_novelty = evaluate_policy(q_network, vae, cost_network, beta, config, tune_train_hpg)
    val_portfolio, val_var, val_novelty = evaluate_policy(q_network, vae, cost_network, beta, config, valid_hpg, tune_train_hpg.iloc[-1])
    train_m = period_metrics(train_portfolio, tune_train_hpg["time"], train_var)
    val_m = period_metrics(val_portfolio, valid_hpg["time"], val_var)
    row = {"stage": TUNING_STAGE, "batch_idx": BATCH_IDX, "config_id": candidate["config_id"], "seed": seed, **config}
    row.update({f"train_{k}": v for k, v in train_m.items()}); row.update({f"val_{k}": v for k, v in val_m.items()})
    row.update({
        "gap_roi": train_m["roi"] - val_m["roi"], "gap_arr": train_m["arr"] - val_m["arr"], "gap_sharpe": train_m["sharpe"] - val_m["sharpe"],
        "q_loss_mean": float(np.mean(q_losses)), "vae_loss_mean": float(np.mean(vae_losses)), "vae_loss_last": float(vae_losses[-1]),
        "vae_recon_last": float(recon_losses[-1]), "vae_kl_last": float(kl_losses[-1]), "cost_loss_last": float(cost_losses[-1]),
        "novelty_mean": float(np.mean(val_novelty)), "novelty_std": float(np.std(val_novelty)), "novelty_max": float(np.max(val_novelty)),
        "finite": bool(np.isfinite(np.asarray(list(train_m.values()) + list(val_m.values()) + vae_losses + cost_losses)).all()),
    })
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

def append_checkpoint(path: Path, row: Mapping[str, Any]) -> None:
    pd.DataFrame([row]).to_csv(path, mode="a", header=not path.exists(), index=False)
    with path.with_suffix(".jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, default=float) + "\n")

RESULTS_CSV = OUTPUT_DIR / f"tim_tham_so_{TUNING_STAGE.lower()}_batch{BATCH_IDX}.csv"
ERRORS_CSV = OUTPUT_DIR / f"tim_tham_so_{TUNING_STAGE.lower()}_batch{BATCH_IDX}_errors.csv"
completed = set()
if RESUME and RESULTS_CSV.exists():
    old = pd.read_csv(RESULTS_CSV); completed = set(zip(old["config_id"].astype(str), old["seed"].astype(int)))

if RUN_TUNING:
    total = len(BATCH_CONFIGS) * len(SCREENING_SEEDS)
    bar = tqdm(total=total, desc=f"Tuning {TUNING_STAGE} batch {BATCH_IDX}/3", unit="run", dynamic_ncols=True, leave=True, mininterval=1.0)
    for candidate in BATCH_CONFIGS:
        for seed in SCREENING_SEEDS:
            key = (candidate["config_id"], int(seed))
            if key in completed:
                bar.update(1); bar.set_postfix(config=candidate["config_id"], seed=seed, status="skip"); continue
            try:
                row = run_config_seed(candidate, seed, bar); append_checkpoint(RESULTS_CSV, row)
                bar.update(1); bar.set_postfix(config=candidate["config_id"], seed=seed, val_sharpe=f"{row['val_sharpe']:.3f}")
            except Exception as error:
                append_checkpoint(ERRORS_CSV, {"stage": TUNING_STAGE, "batch_idx": BATCH_IDX, "config_id": candidate["config_id"],
                                               "seed": seed, "error_type": type(error).__name__, "error": str(error), "traceback": traceback.format_exc()})
                bar.close(); raise
    bar.close(); print("Đã lưu:", RESULTS_CSV)
else:
    print("RUN_TUNING=False — đổi thành True khi đã bật GPU Kaggle.")

In [ ]:
# BLOCK 6 — Báo cáo riêng cho batch trên tài khoản hiện tại (không gộp máy khác)
if not RESULTS_CSV.exists():
    print(f"Chưa có kết quả tại {RESULTS_CSV}. Hãy đặt RUN_TUNING=True và chạy BLOCK 5.")
else:
    batch_results = pd.read_csv(RESULTS_CSV).drop_duplicates(["config_id", "seed"], keep="last")
    param_columns = (
        ["vae_latent_dim", "vae_lr", "vae_beta_kl", "vae_batch_size", "bootstrap_trajectories",
         "bootstrap_updates", "online_aux_updates", "vae_replay_capacity"]
        if TUNING_STAGE == "VAE" else ["beta_0", "beta_decay", "beta_min"]
    )
    leaderboard = batch_results.groupby(["config_id", *param_columns], dropna=False).agg(
        seeds=("seed", "nunique"),
        val_profit_mean=("val_profit", "mean"),
        val_roi_mean=("val_roi", "mean"),
        val_arr_mean=("val_arr", "mean"),
        val_sharpe_mean=("val_sharpe", "mean"),
        val_sharpe_std=("val_sharpe", "std"),
        val_mdd_mean=("val_max_drawdown", "mean"),
        val_violations_mean=("val_violations", "mean"),
        train_sharpe_mean=("train_sharpe", "mean"),
        gap_sharpe_mean=("gap_sharpe", "mean"),
        gap_arr_mean=("gap_arr", "mean"),
        vae_loss_last_mean=("vae_loss_last", "mean"),
        novelty_mean=("novelty_mean", "mean"),
        novelty_std=("novelty_std", "mean"),
        finite_rate=("finite", "mean"),
    ).reset_index()
    leaderboard["score"] = (
        leaderboard["val_sharpe_mean"]
        - 0.25 * leaderboard["val_sharpe_std"].fillna(0.0)
        + 0.02 * leaderboard["val_arr_mean"]
        - 0.01 * leaderboard["val_mdd_mean"]
        - 0.20 * leaderboard["gap_sharpe_mean"].abs()
        - 0.002 * leaderboard["gap_arr_mean"].abs()
        - 0.02 * leaderboard["val_violations_mean"]
    )
    incomplete = (leaderboard["seeds"] < len(SCREENING_SEEDS)) | (leaderboard["finite_rate"] < 1.0)
    leaderboard.loc[incomplete, "score"] = -np.inf
    leaderboard = leaderboard.sort_values("score", ascending=False).reset_index(drop=True)

    leaderboard_csv = OUTPUT_DIR / f"tim_tham_so_{TUNING_STAGE.lower()}_batch{BATCH_IDX}_leaderboard.csv"
    summary_json = OUTPUT_DIR / f"tim_tham_so_{TUNING_STAGE.lower()}_batch{BATCH_IDX}_summary.json"
    plot_path = OUTPUT_DIR / f"tim_tham_so_{TUNING_STAGE.lower()}_batch{BATCH_IDX}_diagnostics.png"
    leaderboard.to_csv(leaderboard_csv, index=False)

    top3 = leaderboard.head(3)
    summary_json.write_text(json.dumps({
        "stage": TUNING_STAGE,
        "batch_idx": BATCH_IDX,
        "configs_assigned": len(BATCH_CONFIGS),
        "configs_completed": int((leaderboard["seeds"] == len(SCREENING_SEEDS)).sum()),
        "screening_seeds": list(SCREENING_SEEDS),
        "score_formula": "Sharpe/ARR/MDD + variance + generalization-gap penalties",
        "top3_in_this_batch": json.loads(top3.to_json(orient="records")),
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"### {TUNING_STAGE} — kết quả riêng Batch {BATCH_IDX}/3")
    print(f"Configs đủ {len(SCREENING_SEEDS)} seed: {(leaderboard.seeds == len(SCREENING_SEEDS)).sum()}/{len(BATCH_CONFIGS)}")
    display_columns = ["config_id", *param_columns, "score", "val_sharpe_mean", "val_arr_mean",
                       "val_mdd_mean", "gap_sharpe_mean", "seeds"]
    try:
        print(top3[display_columns].to_markdown(index=False))
    except ImportError:
        print(top3[display_columns].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].errorbar(np.arange(len(leaderboard)), leaderboard["val_sharpe_mean"],
                     yerr=leaderboard["val_sharpe_std"].fillna(0), fmt="o", ms=4)
    axes[0].set(title=f"Batch {BATCH_IDX} — Validation Sharpe", xlabel="Batch rank", ylabel="Sharpe")
    axes[1].scatter(leaderboard["train_sharpe_mean"], leaderboard["val_sharpe_mean"],
                    c=leaderboard["score"], cmap="viridis")
    bounds = leaderboard[["train_sharpe_mean", "val_sharpe_mean"]]
    low, high = bounds.min().min(), bounds.max().max()
    axes[1].plot([low, high], [low, high], "--", color="gray")
    axes[1].set(title="Generalization Gap", xlabel="Train Sharpe", ylabel="Validation Sharpe")
    axes[2].scatter(leaderboard["val_mdd_mean"], leaderboard["val_arr_mean"],
                    c=leaderboard["score"], cmap="viridis")
    axes[2].set(title="Validation Risk–Return", xlabel="Max Drawdown (%)", ylabel="ARR (%)")
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(plot_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("\n=== FILE CỦA TÀI KHOẢN NÀY ===")
    for path in (RESULTS_CSV, RESULTS_CSV.with_suffix(".jsonl"), leaderboard_csv, summary_json, plot_path):
        print("-", path)